# 01 - Corpus A and D collection

Runs the fetchers on Colab rather than the laptop: better network, and the gigabytes stay
off a machine with 0.77 GB of free RAM.

Everything here is **resumable**. Colab kills sessions at 12 h and after 90 min idle;
rerun the same cell and it picks up where it stopped.

Rate limits are not negotiable - the OSM API is donated infrastructure. The shared token
bucket in `mapguard.data.http` enforces them globally, so do not raise `--rate` to
"go faster".

In [ ]:
# Thin by design. Everything real lives in src/mapguard/ - see CONTRIBUTING.md.
!git clone -q https://github.com/ayushmgarg/osm_mapguard.git mapguard 2>/dev/null || (cd mapguard && git pull -q)
%cd mapguard
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, "src")
print(open("results/gate1a_adiff_batch.json").read()[:200])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

# Bulk data never goes in the repo. On Colab it goes to Drive so it survives the
# 12 h session limit and the 90 min idle kill.
DATA = Path("/content/drive/MyDrive/mapguard")
DATA.mkdir(parents=True, exist_ok=True)
os.environ["MAPGUARD_DATA"] = str(DATA)
os.environ["MAPGUARD_CONTACT"] = "ayushmgarg10@gmail.com"   # goes in the User-Agent
print("MAPGUARD_DATA =", DATA)

In [ ]:
# OSMCha token. Put it in Colab Secrets (key icon, left sidebar) as OSMCHA_TOKEN.
# Never paste it into a cell - notebooks get committed, shared and screenshotted.
from google.colab import userdata
import os
os.environ["OSMCHA_TOKEN"] = userdata.get("OSMCHA_TOKEN")
print("token loaded:", bool(os.environ.get("OSMCHA_TOKEN")))

## Corpus A - metadata

~18,276 changesets at 3/s. About 100 minutes.

In [ ]:
!python scripts/fetch_corpus_a_meta.py --rate 3 --workers 3

## Corpus A - contents (osmChange)

The edits themselves. Measured at 1.2 s per call.

In [ ]:
!python -m mapguard.data.fetch_corpus_a --rate 3 --workers 3

## Corpus A - geometry via Overpass adiff

**Check `results/gate1b_overpass_capacity.json` first.** Overpass runs two slots per
client and is much slower than the OSM API; the gate says whether the full corpus is
affordable or whether to subset.

If subsetting: sample by changeset id, never by size or user, or the subset correlates
with the label.

In [ ]:
!python scripts/fetch_corpus_a_geometry.py

## Corpus A - user histories, point-in-time

One call per user, not per changeset.

In [ ]:
!python scripts/fetch_user_histories.py --rate 3 --workers 3

## Corpus D - live 2026

The temporal test set. State the exact window; it goes in the paper.

In [ ]:
!python scripts/harvest_corpus_d.py --minutes 1440

In [ ]:
!python scripts/fetch_osmcha_labels.py --since 2026-01-01 --max-pages 400

## Build the feature store

Three Parquet tables. Schema frozen end of week 1.

In [ ]:
!python -m mapguard.features.build_store

In [ ]:
import pyarrow.parquet as pq, os
t = pq.read_table(os.environ["MAPGUARD_DATA"] + "/interim/changesets.parquet")
print(t.num_rows, "changesets x", t.num_columns, "features")
print(open("results/../" + "data/interim/build_report.json").read() if False else "")